**Bakehouse Transactions**

Dataset: samples.bakehouse.sales_transactions

Difficulty: Easy

Topics: aggregation, distinct, date, groupBy, F.min, F.max

In [0]:
from pyspark.sql import functions as F, types as t

- Learn — Aggregations and Date Extraction

- Function	What it does

- F.sum("col")	Sums all values in a column
- F.count("*")	Counts all rows (including nulls)
- F.countDistinct("col")	Counts unique non-null values
- F.min("col")	Returns the smallest value — works on dates and timestamps too
- F.max("col")	Returns the largest value — gives the most recent date/timestamp
- F.year(col)	Extracts the year from a date/timestamp
- F.month(col)	Extracts the month number (1–12)
- F.to_date(col)	Casts a string/timestamp to DateType
- .alias("new_name")	Renames the output column

In [0]:
# Run this example first -- then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_transactions")

# Aggregate across the whole table
df.agg(
    F.count("*").alias("num_transactions"),
    F.countDistinct("customerID").alias("unique_customers"),
    F.round(F.avg("totalPrice"), 2).alias("avg_order_value")
).show()

# Extract year and month from the dateTime column
df.select(
    F.year("dateTime").alias("yr"),
    F.month("dateTime").alias("mo")
).groupBy("yr", "mo").count().orderBy("yr", "mo").show(5)

**Problem 1**

Calculate the total number of transactions and the total revenue across all sales. Load samples.bakehouse.sales_transactions and return a single row.

Expected output columns:

- total_transactions - count of all transaction records
- total_revenue - sum of totalPrice across all transactions

In [0]:
df.printSchema()

In [0]:
result_1= df.agg(F.count("transactionID").alias("total_transactions"), F.sum("totalPrice").alias("total_revenue"))

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'total_transactions' in cols, "Missing column: total_transactions"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_1.collect()[0]
assert row['total_transactions'] > 0, "total_transactions must be > 0"
assert row['total_revenue'] > 0, "total_revenue must be > 0"
print(f"Problem 1 passed ✓  ({cnt} rows, transactions={row['total_transactions']}, revenue={row['total_revenue']})")

**Problem 2**

List all unique products sold by the bakehouse, sorted alphabetically. Each product name should appear exactly once.

Expected output columns:

product - unique product name (sorted A → Z)

In [0]:
result_2=df.select("product").distinct().orderBy(F.col("product").asc())
result_2.show()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'product' in cols, "Missing column: product"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
products = [r['product'] for r in result_2.collect()]
assert products == sorted(products), "Products must be sorted alphabetically (ascending)"
assert len(products) == len(set(products)), "Product names must be distinct"
print(f"Problem 2 passed ✓  ({cnt} rows)")